####  B Vidakovic Quantum Framework for Wavelet Shrinkage 
## Example 1

# Quantum Daubechies DAUB2 Multi-level Transform as a Single Gate
### (Qiskit 2.x)

This notebook:
- uses orthonormal DB2 coefficients  

$h = \left(
\frac{1 + \sqrt{3}}{4\sqrt{2}},
\frac{3 + \sqrt{3}}{4\sqrt{2}},
\frac{3 - \sqrt{3}}{4\sqrt{2}},
\frac{1 - \sqrt{3}}{4\sqrt{2}}
\right)
$

- supports circular `shift` (0..3),
- supports `levels` = number of wavelet decomposition levels, 1,2, are 3.
- amplitude-initializes $y=(1,0,-3,2,1,0,1,2)$ 
- applies the resulting unitary $W$ (size 8×8 for 3 qubits) and compares quantum vs classical results.

We use Qiskit 2.x primitives: `Initialize` and `UnitaryGate`.



In [3]:
# Imports
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import Initialize
from qiskit.quantum_info import Operator
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator

np.set_printoptions(precision=6, suppress=True)


In [4]:
#  DB2 (D4) coefficients (orthonormal normalization)
sqrt3 = np.sqrt(3.0)
den = 4.0 * np.sqrt(2.0)   # <-- IMPORTANT: 4*sqrt(2), not 2*sqrt(2)

h = np.array([
    (1.0 + sqrt3) / den,
    (3.0 + sqrt3) / den,
    (3.0 - sqrt3) / den,
    (1.0 - sqrt3) / den
], dtype=np.float64)

L = len(h)
# detail (wavelet) filter g_k = (-1)^k * h_{L-1-k}
g = np.array([((-1)**k) * h[L - 1 - k] for k in range(L)], dtype=np.float64)

print("h =", h)
print("g =", g)
print("sum h^2 =", np.sum(h**2))   # should be ~1



h = [ 0.482963  0.836516  0.224144 -0.12941 ]
g = [-0.12941  -0.224144  0.836516 -0.482963]
sum h^2 = 0.9999999999999997


In [5]:
# Build single-level W (size N) with shift
def build_wavelet_matrix_level(N, h, g, shift=0):
    if N % 2 != 0:
        raise ValueError("N must be even")
    L = len(h)
    half = N // 2
    W = np.zeros((N, N), dtype=np.float64)
    for k in range(half):
        for j in range(L):
            idx = (2*k + shift + j) % N
            W[k, idx] = h[j]
        for j in range(L):
            idx = (2*k + shift + j) % N
            W[half + k, idx] = g[j]
    return W

In [6]:
# Build multi-level W
def build_wavelet_matrix(N, h, g, shift=0, levels=1):
    W_total = np.eye(N)
    size = N
    for lev in range(levels):
        W_level = build_wavelet_matrix_level(size, h, g, shift)
        # Embed into block diag: W_level ⊕ I_rest
        top = W_level
        bottom = np.eye(N - size)
        W_embedded = np.block([
            [top, np.zeros((size, N-size))],
            [np.zeros((N-size, size)), bottom]
        ])
        W_total = W_embedded @ W_total
        size //= 2
        if size < 2:
            break
    return W_total

In [7]:
# Example: N=8, shift=-1, levels=2
N = 8
W = build_wavelet_matrix(N, h, g, shift=-1, levels=2)

# Check unitarity
print("Max deviation from I:", np.max(np.abs(W.T @ W - np.eye(N))))
# Print matrix W
print(W)

Max deviation from I: 6.661338147750939e-16
[[ 0.63726   0.295753  0.079247 -0.01226  -0.13726   0.204247  0.420753
   0.51226 ]
 [-0.13726   0.204247  0.420753  0.51226   0.63726   0.295753  0.079247
  -0.01226 ]
 [-0.170753  0.353766  0.728766 -0.045753 -0.51226  -0.170753 -0.045753
  -0.13726 ]
 [-0.51226  -0.170753 -0.045753 -0.13726  -0.170753  0.353766  0.728766
  -0.045753]
 [-0.224144  0.836516 -0.482963  0.        0.        0.        0.
  -0.12941 ]
 [ 0.       -0.12941  -0.224144  0.836516 -0.482963  0.        0.
   0.      ]
 [ 0.        0.        0.       -0.12941  -0.224144  0.836516 -0.482963
   0.      ]
 [-0.482963  0.        0.        0.        0.       -0.12941  -0.224144
   0.836516]]


In [8]:
# Input vector y
y = np.array([1.0, 0.0, -3.0, 2.0, 1.0, 0.0, 1.0, 2.0], dtype=np.float64)
norm_y = np.linalg.norm(y)
y_norm = y / norm_y

# Classical wavelet transform
classical_result = W @ y_norm
print("Classical result:\n", classical_result)

# Original scaling
print("Original Scaling:\n", classical_result * norm_y)

Classical result:
 [ 0.376333  0.070881 -0.733674 -0.040923  0.215988  0.416468 -0.215988
  0.215988]
Original Scaling:
 [ 1.683013  0.316987 -3.281089 -0.183013  0.965926  1.862501 -0.965926
  0.965926]


In [9]:
# Quantum circuit: amplitude initialize y_norm, then apply W
num_qubits = 3
init = Initialize(y_norm)

qc = QuantumCircuit(num_qubits)
qc.append(init, qc.qubits)

U = Operator(W)
unitary_gate = UnitaryGate(U.data, label="DAUB2_W")
qc.append(unitary_gate, qc.qubits)
qc.save_statevector()
qc.draw('text')
print(qc.draw(output="text"))

# 2) Matplotlib figure (best for PNG/PDF export)
fig = qc.draw(output="mpl")          # or: circuit_drawer(qc, output="mpl")
fig.savefig("e01circuit.png", dpi=300, facecolor="white", transparent=False)
fig.savefig("e01circuit.pdf", dpi=300, facecolor="white", transparent=False)


     ┌───────────────────────────────────────────────────────────────────┐»
q_0: ┤0                                                                  ├»
     │                                                                   │»
q_1: ┤1 Initialize(0.22361,0,-0.67082,0.44721,0.22361,0,0.22361,0.44721) ├»
     │                                                                   │»
q_2: ┤2                                                                  ├»
     └───────────────────────────────────────────────────────────────────┘»
«     ┌──────────┐ statevector 
«q_0: ┤0         ├──────░──────
«     │          │      ░      
«q_1: ┤1 DAUB2_W ├──────░──────
«     │          │      ░      
«q_2: ┤2         ├──────░──────
«     └──────────┘      ░      


In [10]:
# Simulate
sim = AerSimulator()
tqc = transpile(qc, sim)
result = sim.run(tqc).result()
statevec = result.get_statevector(qc)

print("Quantum statevector:\n", np.array(statevec))
print("\nMax abs diff quantum vs classical:", np.max(np.abs(statevec - classical_result)))


Quantum statevector:
 [ 0.376333+0.j  0.070881+0.j -0.733674+0.j -0.040923+0.j  0.215988+0.j
  0.416468+0.j -0.215988+0.j  0.215988+0.j]

Max abs diff quantum vs classical: 1.1102230246251565e-16


In [11]:
# Compare details
print("Classical result (real):\n", classical_result)
print("\nClasssical result scaled:\n", classical_result * norm_y)
print("\nQuantum statevector (real part):\n", np.real(statevec))
print("\nImag parts (should be ~0):\n", np.imag(statevec))


Classical result (real):
 [ 0.376333  0.070881 -0.733674 -0.040923  0.215988  0.416468 -0.215988
  0.215988]

Classsical result scaled:
 [ 1.683013  0.316987 -3.281089 -0.183013  0.965926  1.862501 -0.965926
  0.965926]

Quantum statevector (real part):
 [ 0.376333  0.070881 -0.733674 -0.040923  0.215988  0.416468 -0.215988
  0.215988]

Imag parts (should be ~0):
 [0. 0. 0. 0. 0. 0. 0. 0.]


## Wavmat check
### The same result is obtained by MATLAB's Wavmat(h, 8, 2, 3).
Shift here (-1) corresponds to  Wavmat's (3). The first row of the submatrix
$G_1$ in W is $(-h_2 \; h_1 \; -h_0 \; 0 \; 0 \; 0 \;  0 \; h_3)$ which is $(g_1 \; g_2\;  g_3 \; 0 \; 0 \; 0 \; 0 \; g_0).$
This is the same result obtained by Mallat's algorithm as in the monograph by Vidakovic (1999) pages 116-117.